# Adpating TEMPO to GluonTS

This notebook adapts [TEMPO](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://arxiv.org/pdf/2310.04948) to [GluonTS](https://ts.gluon.ai/stable/index.html) and follows [this tutorial](https://ts.gluon.ai/stable/tutorials/advanced_topics/howto_pytorch_lightning.html). There are three main tasks we need to complete to adapt TEMPO to GluonTS:
1. Creating a [custom dataset](https://ts.gluon.ai/stable/tutorials/forecasting/quick_start_tutorial.html#Custom-datasets) that follows GluonTS's format
2. Creating a [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) wrapper around TEMPO so we can train the model
3. Creating a GluonTS [`Predictor`](https://ts.gluon.ai/stable/api/gluonts/gluonts.torch.model.predictor.html?highlight=predictor#module-gluonts.torch.model.predictor) so we can perform inference

## [Datasets](https://ts.gluon.ai/stable/tutorials/forecasting/quick_start_tutorial.html#Datasets)

### [Built-in Datasets](https://ts.gluon.ai/stable/tutorials/forecasting/quick_start_tutorial.html#Provided-datasets)

GluonTS comes with a number of publicly available built-in datasets. Here are the names of the datasets they provide

In [61]:
from gluonts.dataset.repository import dataset_names

print(f"Available datasets: {dataset_names}")

Available datasets: ['constant', 'exchange_rate', 'solar-energy', 'electricity', 'traffic', 'exchange_rate_nips', 'electricity_nips', 'traffic_nips', 'solar_nips', 'wiki2000_nips', 'wiki-rolling_nips', 'taxi_30min', 'kaggle_web_traffic_with_missing', 'kaggle_web_traffic_without_missing', 'kaggle_web_traffic_weekly', 'm1_yearly', 'm1_quarterly', 'm1_monthly', 'nn5_daily_with_missing', 'nn5_daily_without_missing', 'nn5_weekly', 'tourism_monthly', 'tourism_quarterly', 'tourism_yearly', 'cif_2016', 'london_smart_meters_without_missing', 'wind_farms_without_missing', 'car_parts_without_missing', 'dominick', 'fred_md', 'pedestrian_counts', 'hospital', 'covid_deaths', 'kdd_cup_2018_without_missing', 'weather', 'm3_monthly', 'm3_quarterly', 'm3_yearly', 'm3_other', 'm4_hourly', 'm4_daily', 'm4_weekly', 'm4_monthly', 'm4_quarterly', 'm4_yearly', 'm5', 'uber_tlc_daily', 'uber_tlc_hourly', 'airpassengers', 'australian_electricity_demand', 'electricity_hourly', 'electricity_weekly', 'rideshare_wit

We can download one of the built-in datasets using the `get_dataset()` method

In [62]:
from gluonts.dataset.repository import get_dataset

dataset = get_dataset("m4_hourly")
print(f'Type: {type(dataset)}')

Type: <class 'gluonts.dataset.common.TrainDatasets'>


GluonTS [`TrainDatasets`](https://ts.gluon.ai/stable/api/gluonts/gluonts.dataset.common.html?highlight=traindataset#gluonts.dataset.common.TrainDatasets) are objects that consist of three main members:
- `dataset.train` 
- `dataset.test`
- `dataset.metadata`

`dataset.train` is an iterable collection of data entries used for training.
- Each entry corresponds to a single time series containing the following fields:
    - `target`: An array of time series values
    - `start`: The starting timestamp of the time series.

In [63]:
from gluonts.dataset.util import to_pandas

# Create an iterator over dataset.train
train_iterator = iter(dataset.train)

# Get the first time series in dataset.train
training_time_series = next(train_iterator)

# Create a Pandas series of the first time series
train_series = to_pandas(training_time_series)

# Get the first time series's values
values = training_time_series["target"]
print(f'target[:10] = {values[:10]}')

# Get the first time series's starting timestamp
start = training_time_series["start"]
print(f'start = {start}')

target[:10] = [605. 586. 586. 559. 511. 443. 422. 395. 382. 370.]
start = 1750-01-01 00:00


Similar to `dataset.train`, `dataset.test` is an iterable collection of data entries used for inference.
- Each entry in dataset.test is an extended version of the corresponding entry in dataset.train, containing additional time steps at the end of the series. 
- This extension, known as the forecasting window, has a length equal to the recommended prediction length and represents the period that the model aims to forecast.

In [64]:
# Create an iterator over dataset.test
test_iterator = iter(dataset.test)

# Get the first time series in dataset.test
test_time_series = next(test_iterator)

# Create a Pandas series of the first time series
test_series = to_pandas(test_time_series)

# Get the first time series's values
values = test_time_series["target"]
print(f'target[:10] = {values[:10]}')

# Get the first time series's starting timestamp
start = test_time_series["start"]
print(f'start = {start}')

# Get the forecasting window's length for the first time series
forecasting_window_length = len(test_series) - len(train_series)
print(f'Forecasting window length: {forecasting_window_length}')

target[:10] = [605. 586. 586. 559. 511. 443. 422. 395. 382. 370.]
start = 1750-01-01 00:00
Forecasting window length: 48


- `dataset.metadata` contains metadata of the dataset such as the frequency of the time series, a recommended prediction horizon, associated features, etc.

In [65]:
# Number of future time steps to predict values for 
prediction_length = dataset.metadata.prediction_length
print(f'Prediction length: {prediction_length}')

# How often values are recorded in the time series
frequency = dataset.metadata.freq
print(f'Frequency: {frequency}')

Prediction length: 48
Frequency: H


### [Custom Datasets](https://ts.gluon.ai/stable/tutorials/forecasting/quick_start_tutorial.html#Custom-datasets)

GluonTS doesn't require custom datasets to use this specific format. The only requirements for a custom dataset are
- be iterable
- have a `target` and `start` field

#### [Dummy Dataset](https://ts.gluon.ai/stable/tutorials/forecasting/quick_start_tutorial.html#Custom-datasets)

Here's an example of creating a custom GluonTS dataset using randomly generated data. The dataset's values will be in a `numpy.array` and the dataset's indices (i.e. timestamps) will be in a `pandas.Period`.

First, we'll define the dataset's metadata and create the values and indices.

In [66]:
import numpy as np
import pandas as pd

# Number of time series to store in dummy dataset
num_time_series = 256  * 2

# Number of time steps in each time series
num_time_steps = 336 * 2

# Define each time series's frequency
freq = "1H"

# Create randomy generated values to use in the dummy dataset
dummmy_values = np.random.normal(size=(num_time_series, num_time_steps))
print(f'dummy_values.shape: {dummmy_values.shape}')

# Create the starting timestamp of the dummy dataset
start = pd.Period("01-01-2019", freq=freq)
print(f'start: {start}')

dummy_values.shape: (512, 672)
start: 2019-01-01 00:00


Then, we'll split the dataset and bring it into a GluonTS appropriate form. And that's it! We now have a our dataset in a GluonTS appropriate form. 

In [67]:
from gluonts.dataset.common import ListDataset

# Remove the prediction window from the data in the training set
train_data = dummmy_values[:, :-prediction_length]

# Create the training set
train_set = ListDataset(
    [{"target": time_series, "start": start} for time_series in train_data],
    freq=freq
)

# Create the test set and include the prediction window
test_set = ListDataset(
    [{"target": time_series, "start": start} for time_series in dummmy_values],
    freq=freq
)

#### Custom Datasets for TEMPO

Now that we know how to create custom GluonTS datasets using dummy data, let's create custom GluonTS datasets that we can use with TEMPO.

In [68]:
# Goal: Create GluonTS datasets where each time series has shape (1, 336, 1)

## Training

### PyTorch Lightning Wrapper

Define hyperparameters for training 

In [69]:
learning_rate = 1e-3
batch_size = 128
num_batches_per_epoch = 50
max_epochs = 1
context_length = 2 * 7 * 24
prediction_length = 96

Create a [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) wrapper to wrap around TEMPO so we can train the model using PyTorch Lightning

In [78]:
import pytorch_lightning as pl
import torch
from gluonts.torch import PyTorchPredictor
from gluonts.torch.distributions import StudentTOutput
from gluonts.model.forecast_generator import DistributionForecastGenerator

from tempo.models.TEMPO import TEMPO

class LightningTEMPO(TEMPO, pl.LightningModule):
    def __init__(self, configs, args=None):
        super().__init__(configs)
        # TODO: change args default value so it's not None once you finish your prototype
        # Commmand line arguments
        self.args = args  

        # Model configuration
        self.configs = configs

        # TODO: once you get a prototype working, change the code to allow for different output distributions
        # Type of distribution for model's output. We'll use a Student's t-distribution
        self.distr_output = StudentTOutput()

    def training_step(self, batch, batch_index):
        """
        Defines the logic for a single training loop iteration.
        """       
        # Get past time series values
        past_target = batch["past_target"]

        # Get future time series values
        future_target = batch["future_target"]
        
        # TODO: figure out how to get trend, seasonal, and residual components from custom GluonTS datasets
        # Compute forward pass to get Student's t-distribution arguments
        distr_args, local_loss = self(x=past_target)  

        # Create Student's t-distribution
        student_t_distr = self.distr_output.distribution(distr_args)
        
        # TODO: once you get a prototype working, change the code to compute different losses based on output distribution
        # Compute Student's t negative log-likelihood loss
        loss = -student_t_distr.log_prob(future_target)

        return loss.mean()

    # TODO: 
    def validation_step(self, batch, batch_index):
        """
        Defines the logic for a single validation loop iteration.
        """
        pass

    # TODO:
    def test_step(self, batch, batch_index):
        """
        Defines the logic for a single test loop iteration.
        """
        pass

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)
        return optimizer

    def get_predictor(self, input_transform,):
        """
        Returns predictor for performing inference.
        """
        return PyTorchPredictor(
            prediction_length=prediction_length,  
            input_names=["past_target"],  
            prediction_net=super,  
            batch_size=batch_size, 
            input_transform=input_transform, 
            forecast_generator=DistributionForecastGenerator(self.distr_output)
        )


### Training Dataloader

Here, we'll create a dataloader for the training set using the built-in GluonTS dataset that was loaded earlier

In [71]:
from gluonts.dataset.field_names import FieldName
from gluonts.transform import (
    AddObservedValuesIndicator,
    InstanceSplitter,  
    ExpectedNumInstanceSampler,  
)

Impute `nan`s in the target field with 0 and add a field indicating which values were imputed

In [72]:
mask_unobserved = AddObservedValuesIndicator(
    target_field=FieldName.TARGET,
    output_field=FieldName.OBSERVED_VALUES,
)

Split instances in the trianing set

In [73]:
instannce_sampler = ExpectedNumInstanceSampler(
        num_instances=1,
        min_future=prediction_length,
    )

# split instances in training set
training_splitter = InstanceSplitter(
    target_field=FieldName.TARGET, 
    is_pad_field=FieldName.IS_PAD,  
    start_field=FieldName.START, 
    forecast_start_field=FieldName.FORECAST_START,  
    instance_sampler=instannce_sampler,  
    past_length=context_length,  
    future_length=prediction_length, 
    time_series_fields=[FieldName.OBSERVED_VALUES],  
)

Create dataloader for training set

In [74]:
from gluonts.dataset.loader import TrainDataLoader
from gluonts.itertools import Cached
from gluonts.torch.batchify import batchify

data_loader = TrainDataLoader(
    Cached(dataset.train),
    batch_size=batch_size,
    stack_fn=batchify,
    transform=mask_unobserved + training_splitter,
    num_batches_per_epoch=num_batches_per_epoch,
)

Now that we have a PyTorch Lightning wrapper and a training dataloader, we can initialize the model and train it

In [75]:
from omegaconf import OmegaConf
from pytorch_lightning import Trainer

# Load model configuration
configs = OmegaConf.load('./configs/run_TEMPO.yml')

# Initialize model wrapped with PyTorch Lightning
model = LightningTEMPO(configs)

# Initialize PyTorch Lightning trainer
trainer = Trainer(max_epochs=max_epochs)

# Train model
trainer.fit(model, data_loader)

------------------No need to load pretrained GPT model------------------


/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/site-packages/peft/tuners/lora/layer.py:1150: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/mike_gee/miniconda3/envs/tempo/lib/python3.8/s ...
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/si

trainable params: 308736 || all params: 82207488


Training: |          | 0/? [00:00<?, ?it/s]

Creating Student's t-distribution output
distr_args len: 3
distr_args shape: torch.Size([128, 96])
Returning (distr_args, loss_local)
torch.Size([128, 96])
Creating Student's t-distribution output
distr_args len: 3
distr_args shape: torch.Size([128, 96])
Returning (distr_args, loss_local)
torch.Size([128, 96])
Creating Student's t-distribution output
distr_args len: 3
distr_args shape: torch.Size([128, 96])
Returning (distr_args, loss_local)
torch.Size([128, 96])
Creating Student's t-distribution output
distr_args len: 3
distr_args shape: torch.Size([128, 96])
Returning (distr_args, loss_local)
torch.Size([128, 96])
Creating Student's t-distribution output
distr_args len: 3
distr_args shape: torch.Size([128, 96])
Returning (distr_args, loss_local)
torch.Size([128, 96])
Creating Student's t-distribution output
distr_args len: 3
distr_args shape: torch.Size([128, 96])
Returning (distr_args, loss_local)
torch.Size([128, 96])
Creating Student's t-distribution output
distr_args len: 3
distr

INFO: `Trainer.fit` stopped: `max_epochs=1` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


## Inference

### Creating a GluonTS Predictor

To perform inference using a GluonTS model and dataset, we need to call the model's `get_predictor()` method to get a [`Predictor`](https://ts.gluon.ai/stable/api/gluonts/gluonts.torch.model.predictor.html?highlight=predictor#module-gluonts.torch.model.predictor). First, we'll split the instances in the test set

In [79]:
from gluonts.transform import TestSplitSampler  

prediction_splitter = InstanceSplitter(
    target_field=FieldName.TARGET,
    is_pad_field=FieldName.IS_PAD,
    start_field=FieldName.START,
    forecast_start_field=FieldName.FORECAST_START,
    instance_sampler=TestSplitSampler(),
    past_length=context_length,
    future_length=prediction_length,
    time_series_fields=[FieldName.OBSERVED_VALUES],
)

Then, we'll get a predictor from our model and use it to compute forecasts

In [81]:
predictor = model.get_predictor(mask_unobserved + prediction_splitter)

TypeError: get_predictor() takes 1 positional argument but 2 were given

### Model Evaluation

For example, we can do backtesting on the test dataset: in what follows, `make_evaluation_predictions` will slice out the trailing `prediction_length` observations from the test time series, and use the given predictor to obtain forecasts for the same time range.

In [ ]:
from gluonts.evaluation import make_evaluation_predictions, Evaluator

In [ ]:
forecast_it, ts_it = make_evaluation_predictions(
    dataset=dataset.test, predictor=predictor_pytorch
)

forecasts_pytorch = list(f.to_sample_forecast() for f in forecast_it)
tss_pytorch = list(ts_it)

Once we have the forecasts, we can plot them:

In [ ]:
plt.figure(figsize=(20, 15))
date_formater = mdates.DateFormatter("%b, %d")
plt.rcParams.update({"font.size": 15})

for idx, (forecast, ts) in islice(enumerate(zip(forecasts_pytorch, tss_pytorch)), 9):
    ax = plt.subplot(3, 3, idx + 1)
    plt.plot(ts[-5 * prediction_length :].to_timestamp(), label="target")
    forecast.plot()
    plt.xticks(rotation=60)
    ax.xaxis.set_major_formatter(date_formater)

plt.gcf().tight_layout()
plt.legend()
plt.show()

And we can compute evaluation metrics, that summarize the performance of the model on our test data.

In [ ]:
evaluator = Evaluator(quantiles=[0.1, 0.5, 0.9])

In [ ]:
metrics_pytorch, _ = evaluator(tss_pytorch, forecasts_pytorch)
pd.DataFrame.from_records(metrics_pytorch, index=["FeedForward"]).transpose()